# Image CLIP Cosine Similarity Example

This notebook demonstrates image-to-image scene relighting using CLIP image embedding cosine similarity (`ImageImageCLIPLoss`).
The light coefficients are optimized so that the rendered scene's CLIP image embedding matches the embedding of a target reference image. This is more effective when using weights that were trained with the contrastive objective of matching images with similar lighting, as described in [sections 4.3.2](https://scholarsarchive.byu.edu/cgi/viewcontent.cgi?article=12256&context=etd#page=31.76) and [4.4](https://scholarsarchive.byu.edu/cgi/viewcontent.cgi?article=12256&context=etd#page=35.1) of the thesis.

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from torchvision.transforms.v2 import RandomChoice, RandomResizedCrop

from examples.example_scenes import (
    BlenderManScene,
    CandleScene,
    CarScene,
    CarStudioScene,
    DinoScene,
    EinarScene,
    EinarSmallDomeScene,
    FlowerPotScene,
    HouseScene,
    RedCarScene,
    SciFiRobotScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
    SpringScene,
)
from losses.image_image import ImageImageCLIPLoss
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.model.model_utils import create_clip_model_and_tokenizer
from utils.optimize import optimize_with_criterion


In [ ]:
# Select the scene to optimize (uncomment the desired scene)
scene = SciFiRobotScene(device=device)
# scene = SpringScene(device=device)
# scene = CarScene(device=device)
# scene = BlenderManScene(device=device)
# scene = RedCarScene(device=device)
# scene = CandleScene(device=device)
# scene = HouseScene(device=device)
# scene = DinoScene(device=device)
# scene = FlowerPotScene(device=device)
# scene = CarStudioScene(configuration='dome_lights', device=device)
# scene = EinarScene(device=device)
# scene = EinarSmallDomeScene(device=device)
# scene = SpringPortraitScene(device=device)
# scene = SpringPortraitSmallDomeScene(device=device)


Select the target image with which to maximize the cosine similarity of its embedding during lighting optimization:

In [ ]:
target_image_path = ""  # TODO: Update with reference image path

In [ ]:
# Hyperparameters
lr = 0.06
n_iter = 250
global_seed = 2

clip_model_name = "ViT-B-16-SigLIP-512"
clip_pretrained = "webli"
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
fine_tune = "siglip_blend-training-data_64-output-dim.pt"

model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
    clip_model_name,
    device=device,
    pretrained=clip_pretrained,
    fine_tune=fine_tune,
)

criterion = ImageImageCLIPLoss(
    reference_image=target_image_path,
    clip_model=model,
    preprocess=preprocess_eval,
    device=device,
)

title_prefix = f"ImageImageCLIP ({clip_model_name})"

size = model.visual.preprocess_cfg["size"] or (224, 224) # type: ignore

optimize_with_criterion(
    scene,
    lr,
    n_iter,
    criterion,
    starting_multiplier_std=(0.1, 0.1, 0.1),
    output_subdirectory_name="image_clip_cosine_example",
    n_results=1,
    # augmentation=RandomResizedCrop(size=size, scale=(0.3, 1.0), antialias=True), # Augmentation can sometimes improve performance here, but it seems to help the most with text guidance. # type: ignore
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix=title_prefix,
    device=device,
    save_every=50,
    model_name=clip_model_name,
    pretrained_source=fine_tune,
    seed=global_seed,
    show_images_after_augmentation=True,
    save_loss_plot_each_iteration=True,
)
